In [25]:
import os
from dotenv import load_dotenv
from langchain_core.documents import Document

load_dotenv()

content_list = [
    "RAG combines retrieval with generation to enhance the output of language models by leveraging external information.",
    "LLMs are trained to process and generate human-like text, usually based on deep learning architectures like transformers.",
    "Vector search involves finding similarities in high-dimensional space, helping identify the closest matching entries to a query.",
    "Hugging Face provides a large repository of pre-trained models and tools that make it easier to fine-tune and deploy NLP models.",
    "A vector database stores high-dimensional vectors, allowing for fast similarity searches, commonly used in AI and NLP applications.",
    "Optimizing a pipeline involves improving data processing, model training, and inference to increase efficiency and reduce latency.",
    "Fine-tuning adjusts a pre-trained model to perform better on a specific task by training it on a smaller, task-specific dataset."
    
]

langchain_documents = []

for content in content_list:
    langchain_documents.append(
        Document(
            page_content=content,
        )
    )

In [26]:
from langchain_mistralai import ChatMistralAI

llm = ChatMistralAI(
    model="mistral-large-latest", streaming=True, api_key="API_key")

In [ ]:
llm1 = ChatGrog(
    model="", streaming=True, api_key="API_key")

In [1]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


template = """your are an llm give one short and one long answers for the given question

Question: {query}
"""
prompt = ChatPromptTemplate.from_template(template)

qa_chain = prompt | llm | StrOutputParser()

NameError: name 'llm' is not defined

In [29]:

query = "what is Vector search?"

# relevant_docs = retriever.invoke(query)
qa_chain.invoke({ "query": query})

"### Short Answer:\nVector search is a technique used to find similar items in a high-dimensional space by comparing vectors. It's commonly used in applications like recommendation systems, image search, and natural language processing.\n\n### Long Answer:\nVector search is a method employed to retrieve similar items from a dataset by utilizing vector representations of data points. This technique is particularly useful in high-dimensional spaces where traditional search methods may fall short. The process involves transforming data into numerical vectors, where each vector encapsulates the essence of an item (such as text, images, or other types of multimedia). These vectors are then compared using various distance metrics (e.g., Euclidean distance, cosine similarity) to find the closest matches.\n\nVector search is widely applied in various domains including recommendation systems, image search, natural language processing (NLP), and more. For instance, in NLP, words or sentences can

In [20]:
sample_queries = [
   
    "What is Retrieval-Augmented Generation (RAG)?",
    "What is a Large Language Model (LLM)?",
    "How does vector search work in NLP?",
    "What is a vector database?",
    "How do you optimize a machine learning pipeline?",
    "Explain the importance of fine-tuning models.",
    "What are the key differences between PyTorch and TensorFlow?"
]

expected_responses = [
    "RAG combines retrieval with generation to enhance the output of language models by leveraging external information.",
    "LLMs are trained to process and generate human-like text, usually based on deep learning architectures like transformers.",
    "Vector search involves finding similarities in high-dimensional space, helping identify the closest matching entries to a query.",
    "Hugging Face provides a large repository of pre-trained models and tools that make it easier to fine-tune and deploy NLP models.",
    "A vector database stores high-dimensional vectors, allowing for fast similarity searches, commonly used in AI and NLP applications.",
    "Optimizing a pipeline involves improving data processing, model training, and inference to increase efficiency and reduce latency.",
    "Fine-tuning adjusts a pre-trained model to perform better on a specific task by training it on a smaller, task-specific dataset."
    
]

In [21]:
from ragas import EvaluationDataset


dataset = []

for query, reference in zip(sample_queries, expected_responses):
    # relevant_docs = retriever.invoke(query)
    response = qa_chain.invoke({"query": query})
    dataset.append(
        {
            "user_input": query,
            "retrieved_contexts":expected_responses,
            "response": response,
            "reference": reference,
        }
    )

evaluation_dataset = EvaluationDataset.from_list(dataset)

In [22]:
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness

evaluator_llm = LangchainLLMWrapper(llm)

result = evaluate(
    dataset=evaluation_dataset,
    metrics=[ LLMContextRecall(),Faithfulness(), FactualCorrectness()],
    llm=evaluator_llm,
)

result

Evaluating: 100%|██████████| 21/21 [04:01<00:00, 11.52s/it]


{'context_recall': 1.0000, 'faithfulness': 0.1215, 'factual_correctness': 0.1667}